###Type Widening and the Variant Data Type
Changing data type is not allowed in delta table. However, type widening is an option.\
Type widening: changing the data type of an existing column to a broader one, without rewriting any of the data already in the table.

##### Part 1 — Type Widening

In [0]:
USE CATALOG workspace;
USE SCHEMA default;

#####1. Create sensor_readings with INT column

In [0]:
CREATE OR REPLACE TABLE sensor_readings (
  sensor_id     STRING,
  reading_value INT,
  recorded_at   TIMESTAMP
)
USING DELTA
COMMENT 'IoT sensor readings — type widening demo';

#####2. Load initial INT readings

In [0]:
INSERT INTO sensor_readings VALUES
  ('S001', 1842,  '2024-07-01 09:00:00'),
  ('S002', 2103,  '2024-07-01 09:01:00'),
  ('S003', 987,   '2024-07-01 09:02:00'),
  ('S004', 31500, '2024-07-01 09:03:00');

#####3. Write LONG values into INT column WITHOUT type widening enabled
>Now the source system is upgraded — sensors now report higher-precision values that exceed the INT maximum of about 2.1 billion. 

In [0]:
%python
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
from datetime import datetime

schema_long = StructType([
    StructField('sensor_id',     StringType(),    False),
    StructField('reading_value', LongType(),      False),
    StructField('recorded_at',   TimestampType(), False),
])

long_readings = [
    Row(sensor_id='S001',
        reading_value=3_200_000_000,          # exceeds INT max (2,147,483,647)
        recorded_at=datetime(2024, 7, 1, 10, 0, 0)),
    Row(sensor_id='S005',
        reading_value=4_100_000_000,
        recorded_at=datetime(2024, 7, 1, 10, 1, 0)),
]

df_long = spark.createDataFrame(long_readings, schema_long)

# This will FAIL — type widening is not yet enabled
df_long.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('sensor_readings')

#####4. Enable Type Widening

In [0]:
ALTER TABLE sensor_readings
SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true');

>You can use ALTER TABLE to upgrade column from INT to LONG.\
ALTER TABLE sensor_readings
ALTER COLUMN reading_value TYPE LONG;

#####5. Write LONG values with mergeSchema — now succeeds

In [0]:
%python
df_long.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('sensor_readings')

display(spark.table('sensor_readings').orderBy('sensor_id', 'recorded_at'))

####6. Type widening outside the family
>mergeSchema or the ALTER TABLE widens only the following path
* BYTE → SHORT → INT → LONG (integer family)
* FLOAT → DOUBLE (floating-point family)

In [0]:
%python
from pyspark.sql.types import DoubleType

schema_double = StructType([
    StructField('sensor_id',     StringType(),    False),
    StructField('reading_value', DoubleType(),    False),  # DOUBLE — wider than LONG but outside the family
    StructField('recorded_at',   TimestampType(), False),
])

double_readings = [
    Row(sensor_id='S006',
        reading_value=1842.75,                 # fractional precision
        recorded_at=datetime(2024, 7, 2, 9, 0, 0)),
]

df_double = spark.createDataFrame(double_readings, schema_double)

df_double.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('sensor_readings')


### Part 2 — Variant Data Type

#####1. Create iot_events with a VARIANT payload column

In [0]:
CREATE OR REPLACE TABLE iot_events (
  event_id    INT,
  received_at TIMESTAMP,
  payload     VARIANT
)
USING DELTA
COMMENT 'IoT events — variable-schema JSON stored as VARIANT';

#####2. Insert events with varying JSON shapes

In [0]:
INSERT INTO iot_events VALUES
  (1, '2024-07-01 09:00:00',
   PARSE_JSON('{"sensor_id":"S001","temp":23.4,"unit":"C","battery":0.92}')),
  (2, '2024-07-01 09:01:00',
   PARSE_JSON('{"sensor_id":"S002","temp":71.2,"unit":"F"}')),
  (3, '2024-07-01 09:02:00',
   PARSE_JSON('{"sensor_id":"S003","pressure":1013.2,"unit":"hPa","location":{"lat":51.5,"lon":-0.12}}')),
  (4, '2024-07-01 09:03:00',
   PARSE_JSON('{"sensor_id":"S004","temp":19.8,"unit":"C","calibration":{"last_date":"2024-06-01","offset":-0.3}}')),
  (5, '2024-07-01 09:04:00',
   PARSE_JSON('{"sensor_id":"S005","temp":22.1,"unit":"C","battery":0.78,"tags":["indoor","lab"]}'));

#####3. Query with the : operator and :: cast

In [0]:
SELECT
    event_id,
    received_at,
    payload:sensor_id::STRING AS sensor_id,
    payload:temp::DOUBLE AS temp,
    payload:unit::STRING AS unit,
    payload:battery::DOUBLE AS battery
FROM iot_events
ORDER BY event_id;

#####4. Navigate nested objects with chained : operators

In [0]:
SELECT
  event_id,
  payload:sensor_id::STRING               AS sensor_id,
  payload:location:lat::DOUBLE            AS latitude,
  payload:location:lon::DOUBLE            AS longitude,
  payload:calibration:last_date::STRING   AS last_calibrated,
  payload:calibration:offset::DOUBLE      AS cal_offset
FROM iot_events
ORDER BY event_id;

#####5. Filter on a VARIANT field

In [0]:
SELECT
  event_id,
  payload:sensor_id::STRING  AS sensor_id,
  payload:temp::DOUBLE       AS temp
FROM  iot_events
WHERE payload:unit::STRING = 'C'
ORDER BY event_id;

####Summary
Two features. Here is what to take away from each.

Type widening: A Delta 4.0 table feature that lets you promote a column's type to a wider one — INT to LONG, FLOAT to DOUBLE, DATE to TIMESTAMP — without rewriting any Parquet files. Enable it with ALTER TABLE ... SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true'). Then either change types explicitly with ALTER TABLE ... ALTER COLUMN, or let mergeSchema handle it automatically when the source type is wider than the target. Requires DBR 15.4+.\
The right tool when a source system increases its numeric precision and you cannot afford a full table rewrite.

Variant. A native Delta data type for variable-schema JSON. Stores data in a compact binary encoding — more flexible than STRUCT, significantly faster than STRING at scale. Create a VARIANT column and ingest with PARSE_JSON. Query with the : operator, cast with ::, navigate nested objects by chaining :. Missing paths return NULL — no errors. Shredding improves query performance further by storing frequently accessed paths as physical columns — happens automatically in the background on DBR 17.2+.\
Use VARIANT when your JSON schema evolves, varies row to row, or comes from an external system. Use STRUCT when the schema is stable and well-known.